## 환경 설정

In [ ]:
import os, sys
os.chdir(r'C:\Users\gogoc\AppData\Local\Temp\echo-tof-verify')
sys.path.insert(0, '.')

## 2. 4가지 오차 메트릭과 순위

| 메트릭 | 의미 | 계산 |
|--------|------|------|
| `intensity_cluster_error` | 강도 패턴 일치도 | ClusterError (intensity RMS) |
| `rms_error` | 질량 일치도 | RMS mass error (ppm) |
| `rms_w_int_error` | mono 보정 질량 일치도 | RMS w/ mono adjustment |
| `mass_cluster_error` | 복합 지표 | abs(rms) + 100 * cluster |

각 메트릭별로 순위를 매기고, 4개 순위의 합이 `overall_order`가 된다.

```python
# pipeline.py: _update_composition_ranking()
r.overall_order = (
    r.int_cluster_order + r.rms_error_order +
    r.mass_cluster_order + r.rms_w_int_order
)
```

MS/MS 데이터가 있으면 `combined_order`로 확장된다:

```python
r.combined_order = (
    r.msms_order * 4 * (1 - ms_contribution) +
    r.overall_order * ms_contribution
)
```

## 3. 기본 예제: 가상 데이터

먼저 간단한 가상 데이터로 파이프라인을 실행해 본다.

In [ ]:
from echo_tof.pipeline import FormulaFinderPipeline

pipeline = FormulaFinderPipeline()

# Step 1: MS 파라미터
pipeline.init_ms(
    mass_tol_ppm=5.0,
    int_tol=100.0,
    min_comp="",
    max_comp="C50 H200 N10 O10 S5",
    dbe_from=-0.5,
    dbe_to=40.0,
    even_electron=True,
    odd_electron=True,
    common_rules=True,
)
print("Step 1: init_ms() done")

In [ ]:
# Step 2: 실측 패턴 (TNT 부근 가상 데이터)
pipeline.set_pattern(
    pattern_mz=[227.0180, 228.0213, 229.0240],
    pattern_int=[10000.0, 850.0, 200.0],
    use_peak=[True, True, True],
    peak_confidence=[1.0, 1.0, 1.0],
    charge=0,
)
print("Step 2: set_pattern() done")
print(f"  정규화된 강도: {pipeline._pattern_int}")

In [ ]:
# Step 3: 후보 제안
count = pipeline.propose_elemental_compositions()
print(f"Step 3: {count} candidates found\n")

# 결과 출력
print(f"{'#':>3s} {'Formula':18s} {'Error(ppm)':>10s} {'Cluster':>8s} "
      f"{'RMS':>8s} {'MassClust':>9s} {'Overall':>7s}")
print("-" * 68)

# overall_order로 정렬
sorted_results = sorted(pipeline.results, key=lambda r: r.overall_order)
for i, r in enumerate(sorted_results[:10], 1):
    print(f"{i:3d} {r.composition:18s} {r.error_ppm:+10.2f} "
          f"{r.intensity_cluster_error:8.4f} {r.rms_error:8.4f} "
          f"{r.mass_cluster_error:9.4f} {r.overall_order:7d}")

## 4. CompositionResult 상세 분석

최상위 후보의 피크별 상세 정보를 확인한다.

In [ ]:
if sorted_results:
    best = sorted_results[0]
    print(f"Best candidate: {best.composition}")
    print(f"  Monoisotopic mass: {best.monoisotopic_mass:.6f} Da")
    print(f"  Mass error: {best.error_ppm:+.2f} ppm ({best.error_mda:+.3f} mDa)")
    print(f"  RDB: {best.rdb:.1f}")
    print(f"  Even electron: {best.is_even_electron}")
    print()
    print(f"  Peak details:")
    for pd in best.peak_details:
        print(f"    Isotope {pd.isotope_index}: "
              f"theor_mz={pd.theoretical_mz:.4f} "
              f"theor_int={pd.theoretical_intensity:.1f}% "
              f"error={pd.error_ppm:+.2f} ppm "
              f"used={pd.is_used}")

## 5. 실제 데이터: mzML에서 스펙트럼 읽기

실제 TOF-MS 데이터(`20260330_TOFMS.mzML`)에서 스펙트럼을 읽어 파이프라인을 실행한다.

mzML 파싱에는 `pyteomics` 또는 간단한 XML 파싱을 사용한다.

In [ ]:
import xml.etree.ElementTree as ET
import base64, struct, zlib
import numpy as np

def read_mzml_spectrum(path, scan_index=0):
    """mzML에서 지정 스캔의 m/z, intensity 배열을 읽는다."""
    ns = '{http://psi.hupo.org/ms/mzml}'
    tree = ET.parse(path)
    root = tree.getroot()
    spectra = root.findall(f'.//{ns}spectrum')
    if scan_index >= len(spectra):
        raise IndexError(f"Scan {scan_index} not found (total: {len(spectra)})")
    spec = spectra[scan_index]

    arrays = {}
    for bda in spec.findall(f'{ns}binaryDataArrayList/{ns}binaryDataArray'):
        cvs = {cv.get('accession'): cv for cv in bda.findall(f'{ns}cvParam')}
        binary_text = bda.find(f'{ns}binary').text
        if binary_text is None:
            continue
        raw = base64.b64decode(binary_text)
        # zlib 압축 체크
        if 'MS:1000574' in cvs:
            raw = zlib.decompress(raw)
        # 64-bit float
        if 'MS:1000523' in cvs:
            arr = np.frombuffer(raw, dtype=np.float64)
        else:
            arr = np.frombuffer(raw, dtype=np.float32)
        # m/z or intensity
        if 'MS:1000514' in cvs:
            arrays['mz'] = arr
        elif 'MS:1000515' in cvs:
            arrays['intensity'] = arr

    return arrays.get('mz'), arrays.get('intensity'), len(spectra)

mzml_path = r'data/mzml/20260330_TOFMS.mzML'
try:
    mz_arr, int_arr, n_scans = read_mzml_spectrum(mzml_path, scan_index=0)
    print(f"mzML loaded: {n_scans} scans")
    print(f"Scan 0: {len(mz_arr)} data points, m/z range: {mz_arr.min():.2f} - {mz_arr.max():.2f}")
except Exception as e:
    print(f"mzML 로딩 실패: {e}")
    mz_arr = None

## 6. mzML 데이터로 파이프라인 실행

스펙트럼에서 특정 m/z 범위의 동위원소 클러스터를 추출하여 파이프라인에 입력한다.

In [ ]:
def extract_isotope_cluster(mz_arr, int_arr, mono_mz, charge=1, n_peaks=4, window=0.1):
    """m/z 배열에서 동위원소 클러스터 피크를 추출한다."""
    spacing = 1.003355 / abs(charge) if charge != 0 else 1.003355
    cluster_mz = []
    cluster_int = []
    for i in range(n_peaks):
        target = mono_mz + i * spacing
        mask = (mz_arr > target - window) & (mz_arr < target + window)
        if mask.any():
            subset_mz = mz_arr[mask]
            subset_int = int_arr[mask]
            max_idx = subset_int.argmax()
            cluster_mz.append(float(subset_mz[max_idx]))
            cluster_int.append(float(subset_int[max_idx]))
        else:
            break
    return cluster_mz, cluster_int

if mz_arr is not None:
    # 예: reserpine [M+H]+ = 609.2812 근처에서 클러스터 추출
    target_mz = 609.28
    cluster_mz, cluster_int = extract_isotope_cluster(
        mz_arr, int_arr, target_mz, charge=1, n_peaks=4
    )
    if len(cluster_mz) >= 2:
        print(f"추출된 클러스터 (target ~ {target_mz}):")
        for i, (m, inten) in enumerate(zip(cluster_mz, cluster_int)):
            print(f"  Peak {i}: m/z={m:.4f}  intensity={inten:.0f}")
    else:
        # 가장 강한 피크 근처에서 시도
        top_idx = int_arr.argmax()
        target_mz = float(mz_arr[top_idx])
        cluster_mz, cluster_int = extract_isotope_cluster(
            mz_arr, int_arr, target_mz, charge=1, n_peaks=4
        )
        print(f"대체 클러스터 (가장 강한 피크 ~ {target_mz:.4f}):")
        for i, (m, inten) in enumerate(zip(cluster_mz, cluster_int)):
            print(f"  Peak {i}: m/z={m:.4f}  intensity={inten:.0f}")
else:
    # fallback 가상 데이터
    print("mzML 없음 -- 가상 데이터 사용")
    cluster_mz  = [609.2812, 610.2846, 611.2873, 612.2901]
    cluster_int = [10000.0, 3800.0, 780.0, 120.0]

In [ ]:
# 파이프라인 실행
pipe2 = FormulaFinderPipeline()

pipe2.init_ms(
    mass_tol_ppm=5.0,
    int_tol=100.0,
    max_comp="C50 H200 N10 O10 S5",
    dbe_from=-0.5,
    dbe_to=40.0,
    common_rules=True,
)

n_peaks = len(cluster_mz)
pipe2.set_pattern(
    pattern_mz=cluster_mz,
    pattern_int=cluster_int,
    use_peak=[True] * n_peaks,
    peak_confidence=[1.0] * n_peaks,
    charge=1,
)

count2 = pipe2.propose_elemental_compositions()
print(f"후보 수: {count2}\n")

sorted2 = sorted(pipe2.results, key=lambda r: r.overall_order)
print(f"{'#':>3s} {'Formula':22s} {'Error(ppm)':>10s} {'Cluster':>8s} "
      f"{'RMS':>8s} {'Overall':>7s}")
print("-" * 62)
for i, r in enumerate(sorted2[:10], 1):
    print(f"{i:3d} {r.composition:22s} {r.error_ppm:+10.2f} "
          f"{r.intensity_cluster_error:8.4f} {r.rms_error:8.4f} "
          f"{r.overall_order:7d}")

## 7. 순위 시스템 해부

각 후보의 4가지 개별 순위가 어떻게 합산되는지 확인한다.

In [ ]:
if sorted2:
    print(f"{'Formula':22s} {'IntClust':>8s} {'RMS':>5s} {'MassClust':>9s} "
          f"{'RMSwInt':>7s} {'Sum':>5s}")
    print("-" * 60)
    for r in sorted2[:8]:
        print(f"{r.composition:22s} {r.int_cluster_order:8d} {r.rms_error_order:5d} "
              f"{r.mass_cluster_order:9d} {r.rms_w_int_order:7d} "
              f"{r.overall_order:5d}")